# Gradient Boosting for Classification — Notes

## Why Classification Is Different From Regression

In regression, we predict a continuous value directly. In classification, we predict a **probability**, so the model works in **log-odds space** instead of raw probability space — this is what makes classification GB trickier than regression GB.

$$\text{log-odds} = \ln\left(\frac{p}{1-p}\right)$$

---

## Step 1: Initialize the Model — $F_0(x)$

Just like regression, we start with a constant prediction, but here it's a constant **log-odds** value based on the overall class distribution:

$$F_0(x) = \log\text{-odds of the target class} = \ln\left(\frac{\#\text{positive}}{\#\text{negative}}\right)$$

This constant log-odds is then converted to a **probability** using the sigmoid function:

$$p = \frac{1}{1+e^{-F_0(x)}}$$

**Example from the board:**
- `pre1(log-odds) = 0.510826` for every row
- `pre1(probability) = 0.625` for every row (via sigmoid of 0.510826)

So every data point starts with the same predicted probability (0.625), since the model hasn't learned anything yet.

---

## Step 2: Compute Residuals

$$res1 = y_i - p_i$$

This is simply **actual class (0 or 1) − predicted probability**.

| is_placed (y) | pre1 (prob) | res1 |
|---|---|---|
| 0 | 0.625 | -0.625 |
| 1 | 0.625 | 0.375 |

- If actual = 1 and predicted prob = 0.625 → residual = **+0.375**
- If actual = 0 and predicted prob = 0.625 → residual = **−0.625**

This residual behaves like a "difference of probability" — it tells the model how wrong the current probability estimate is for each point.

---

## Step 3: Fit a Regression Tree to the Residuals

A **regression tree** (not classification tree) is trained with `res1` as the target, using features like `cgpa`, `iq`. Splits are chosen to minimize squared error, same as in regression boosting.

**Example tree:**
```
              node#0: cgpa <= 6.375
             /                      \
      node#1 (value=0.375)      node#2: iq <= 132.5
                                 /                  \
                          node#3 (value=0.625)   node#4 (value=0.042)
```

Each leaf's `value` here is just the **average residual** of points landing in it — but this raw average is *not* used directly as the update. It must first be converted, because we're working in log-odds space, not probability space.

---

## Step 4: Transform Leaf Output (Log-Odds Conversion)

Since predictions are combined in **log-odds space**, each leaf's average residual must be converted into a log-odds contribution using this formula:

$$\gamma_{leaf} = \frac{\sum \text{Residuals in leaf}}{\sum \big[\text{PreviousProb} \times (1-\text{PreviousProb})\big]}$$

This is derived from the second-order Taylor approximation of the log-loss function (Newton's method step) — it accounts for the curvature of the loss, not just its slope.

**Example (from board, node with values -0.625 and -0.625... i.e. two points with residual -0.625 each, PreviousProb = 0.625 for both):**

$$\gamma = \frac{-0.625 + -0.625}{[0.625(1-0.625)] + [0.625(1-0.625)]} = \frac{-1.25}{0.46875} \approx -2.66$$

Similarly for another leaf: numerator `0.18`-type values give the other leaf outputs shown on the board (`-2.66`, `0.18`, etc. — these become the actual log-odds updates per leaf).

---

## Step 5: Update the Log-Odds Prediction

$$F_1(x) = F_0(x) + \eta \cdot \gamma_{leaf}$$

where $\eta$ is the learning rate (shrinkage factor). This gives a new **log-odds** prediction (`pre2(log-odds)`) for each point, based on which leaf it falls into.

**Example from the table:**

| cgpa | iq | res1 | leaf_entry1 | pre2(log-odds) | pre2(probability) | res2 |
|------|-----|--------|----|-----------|------------|---------|
| 6.82 | 118 | -0.625 | 3 | -2.159174 | 0.103477 | -0.103477 |
| 6.36 | 125 | 0.375 | 1 | 2.110826 | 0.891951 | 0.108049 |
| 6.39 | 148 | -0.625 | 4 | 0.690826 | 0.666151 | -0.666151 |

### Convert log-odds back to probability (sigmoid):
$$p_2 = \frac{1}{1+e^{-F_1(x)}}$$

This gives `pre2(probability)`.

### Compute new residual for the next round:
$$res2 = y_i - p_2$$

This whole process (fit tree on residuals → compute leaf log-odds via the ratio formula → update F(x) → convert to probability → get new residual) is **repeated for M rounds**.

---

## Step 6: Second Iteration — Building the Next Tree

A new regression tree is fit on `res2` using the same features. This produces a new set of leaves and values:

```
              node#0: cgpa <= 6.995
             /                     \
       node#1: iq <= 136.5          node#2 (value=0.166)
       /                \
  node#3 (value=0.055)  node#4 (value=-0.666)
```

Each leaf value here is again the **raw average residual**, which must go through the same log-odds transformation:

$$\gamma = \frac{\sum res2}{\sum[p_2(1-p_2)]}$$

**Worked example from the board:**
$$\gamma = \frac{-0.66}{0.66 \times (1-0.66)} = \frac{-0.66}{0.2244} \approx -1$$

Wait — more precisely per the board: numerator `-0.66`, denominator `0.66×(1-0.66)=0.2244`, giving a result close to `-1` (board shows ≈ `-0.34`... in general this ratio is computed per leaf using the sum of residuals over the sum of `prob×(1-prob)` for **all points in that leaf**, not a single point).

This new leaf output becomes the update added to $F_1(x)$ to get $F_2(x)$, and the cycle continues.

---

## Full Algorithm Summary (Classification)

1. **Initialize**: $F_0(x) = \ln\left(\frac{P(y=1)}{P(y=0)}\right)$ → convert to probability via sigmoid.
2. **For** $m = 1$ to $M$:
   - Compute residuals: $r_i = y_i - p_i$ (actual − predicted probability)
   - Fit a **regression tree** to the residuals → get terminal leaves.
   - For each leaf, compute the log-odds update:
     $$\gamma_{leaf} = \frac{\sum \text{residuals in leaf}}{\sum [p_i(1-p_i)] \text{ in leaf}}$$
   - Update log-odds: $F_m(x) = F_{m-1}(x) + \eta \cdot \gamma_{leaf}$
   - Convert to probability: $p_m = \text{sigmoid}(F_m(x))$
   - Compute new residuals: $r = y - p_m$ for the next round.
3. **Output**: Final probability = sigmoid of the final accumulated log-odds $F_M(x)$.

---

## Key Points to Remember
- Classification GB works in **log-odds space**; predictions are only converted to probability at the end of each stage (via sigmoid).
- Residual = actual class − predicted probability (not actual − log-odds).
- The tree is still a **regression tree** — it predicts residuals, not classes directly.
- Leaf outputs must be transformed using $\frac{\sum residual}{\sum[p(1-p)]}$ before being added to the log-odds — this comes from a second-order (Newton) approximation of log-loss, and is the classification analogue of "just average the residual" in regression GB.
- Learning rate $\eta$ still controls how much each tree's correction contributes, to reduce overfitting.